In [1]:
! pip install BeautifulSoup4
! pip install selenium


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

In [16]:
# 크롬 WebDriver 객체 생성
wd = webdriver.Chrome()

# 특정 페이지 이동
wd.get("https://www.megabox.co.kr/movie") #메가박스 전체영화 > 박스오피스

time.sleep(1)

In [28]:
#[CODE 1]
def mbx(result, wd):
    # 1) 박스오피스 리스트가 있는 div
    movie_box = (By.XPATH, "/html/body/div[2]/div[2]/div[2]/div/div[4]/ol")

    # 2) 더보기 버튼 자동 클릭 루프
    while True:
        time.sleep(1)
        try:
            more_btn = wd.find_element(By.XPATH, "/html/body/div[2]/div[2]/div[2]/div/div[5]/button")
            if more_btn.is_displayed():
                wd.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break  # 더보기 버튼이 더 이상 없음

    # 3) 더보기 클릭이 모두 끝난 후 전체 HTML 가져오기
    element = wd.find_element(*movie_box)
    html = element.get_attribute("outerHTML")
    soup = BeautifulSoup(html, 'html.parser')

    # 4) 모든 영화 li 수집
    movie_list = soup.select("ul > li")

    print(f"총 {len(movie_list)}개 영화 감지됨")  # 디버깅용

    # 5) 각 영화 정보 파싱
    for movie in movie_list:
        # 등수
        try:
            rank = movie.select_one("p.rank").text.strip()
        except:
            rank = ""

        # 포스터 이미지 src + 제목(alt)
        try:
            img = movie.select_one("img")
            img_src = img['src']
            img_title = img['alt']
        except:
            img_src = ""
            img_title = ""

        # 화면 유형 체크 함수
        def check_exist(id_name):
            tag = movie.select_one(f"div.screen-type2 p#{id_name}")
            return 1 if tag and tag.text.strip() else 0

        mx4d = check_exist("mx4d")
        dolby = check_exist("dolby")
        atmos = check_exist("atmos")

        # 독점 여부
        try:
            mega_only = 1 if movie.select_one("div.megaOnly") else 0
        except:
            mega_only = 0

        # 영화 요약
        try:
            summary = movie.select_one("div.movie-score div.summary").text.strip()
        except:
            summary = ""

        # 관람 평점
        try:
            score = movie.select_one("p.number").text.strip()
        except:
            score = ""

        # 영화 제목
        try:
            title = movie.select_one("div.tit-area p.tit")["title"]
        except:
            title = ""

        # 예매율
        try:
            booking_rate = movie.select_one("div.rate-date span.rate").text.strip()
        except:
            booking_rate = ""

        # 개봉일
        try:
            release_date = movie.select_one("div.rate-date span.date").text.strip()
        except:
            release_date = ""

        # 좋아요 수
        try:
            like = movie.select_one("div.btn-util button.btn-like span").text.strip()
        except:
            like = ""

        # 저장
        result.append([
            rank, img_src, img_title, mx4d, dolby, atmos, mega_only,
            summary, score, title, booking_rate,
            release_date, like
        ])

    return


In [29]:
#[CODE 0]
def main():
    # 1) Webdriver 실행
    wd = webdriver.Chrome()
    wd.get("https://www.megabox.co.kr/movie")
    time.sleep(2)

    result = []
    print('Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>')

    # 2) 박스오피스 크롤링
    mbx(result, wd)

    # 3) DataFrame → CSV 저장
    table = pd.DataFrame(result, columns=[
        'rank', 'img_src', 'img_title', 'mx4d', 'dolby', 'atmos',
        'mega_only', 'summary', 'score', 'title',
        'booking_rate', 'release_date', 'like'
    ])

    table.to_csv('megaboxCrawling.csv', encoding='cp949', mode='w', index=False)
    print('megaboxCrawling.csv  파일저장 완료 >>>>>>>>>>>>>>>>>')


if __name__ == '__main__':
    main()

Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>
총 0개 영화 감지됨
megaboxCrawling.csv  파일저장 완료 >>>>>>>>>>>>>>>>>


In [31]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

#[CODE 1]
def mbx(result, wd):
    # 1) 초기 로딩 시 영화 리스트 <ol>
    initial_movie_box = (By.XPATH, "/html/body/div[2]/div[2]/div[2]/div/div[4]/ol")

    # 2) 더보기 버튼 자동 클릭 루프
    while True:
        time.sleep(1)
        try:
            more_btn = wd.find_element(By.XPATH, "/html/body/div[2]/div[2]/div[2]/div/div[5]/button")
            if more_btn.is_displayed():
                wd.execute_script("arguments[0].click();", more_btn)
                # 새 영화 <li>가 로딩될 때까지 대기
                WebDriverWait(wd, 5).until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='movie-list']/ol/li"))
                )
            else:
                break
        except:
            break  # 더보기 버튼 없음 → 종료

    # 3) 더보기 클릭 완료 후 전체 영화 영역 <div>
    full_movie_box = (By.XPATH, "/html/body/div[2]/div[2]/div[2]/div/div[4]")
    element = wd.find_element(*full_movie_box)
    html = element.get_attribute("outerHTML")
    soup = BeautifulSoup(html, 'html.parser')

    # 4) 모든 영화 <li> 선택
    movie_list = soup.select("ol > li")
    print(f"총 {len(movie_list)}개 영화 감지됨")  # 디버깅용

    # 5) 각 영화 정보 파싱
    for movie in movie_list:
        # 등수
        try:
            rank = movie.select_one("p.rank").text.strip()
        except:
            rank = ""

        # 포스터 이미지 src + 제목(alt)
        try:
            img = movie.select_one("img")
            img_src = img['src']
            img_title = img['alt']
        except:
            img_src = ""
            img_title = ""

        # 화면 유형 체크 함수
        def check_exist(id_name):
            tag = movie.select_one(f"div.screen-type2 p#{id_name}")
            return 1 if tag and tag.text.strip() else 0

        mx4d = check_exist("mx4d")
        dolby = check_exist("dolby")
        atmos = check_exist("atmos")

        # 독점 여부
        try:
            mega_only = 1 if movie.select_one("div.megaOnly") else 0
        except:
            mega_only = 0

        # 영화 요약
        try:
            summary = movie.select_one("div.movie-score div.summary").text.strip()
        except:
            summary = ""

        # 관람 평점
        try:
            score = movie.select_one("p.number").text.strip()
        except:
            score = ""

        # 영화 제목
        try:
            title = movie.select_one("div.tit-area p.tit")["title"]
        except:
            title = ""

        # 예매율
        try:
            booking_rate = movie.select_one("div.rate-date span.rate").text.strip()
        except:
            booking_rate = ""

        # 개봉일
        try:
            release_date = movie.select_one("div.rate-date span.date").text.strip()
        except:
            release_date = ""

        # 좋아요 수
        try:
            like = movie.select_one("div.btn-util button.btn-like span").text.strip()
        except:
            like = ""

        # 결과 저장
        result.append([
            rank, img_src, img_title, mx4d, dolby, atmos, mega_only,
            summary, score, title, booking_rate, release_date, like
        ])


#[CODE 0]
def main():
    # 1) Selenium 드라이버 실행
    wd = webdriver.Chrome()
    wd.get("https://www.megabox.co.kr/movie")
    time.sleep(2)

    result = []
    print("Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>")

    # 2) 영화 크롤링
    mbx(result, wd)

    # 3) DataFrame → CSV 저장
    table = pd.DataFrame(result, columns=[
        'rank', 'img_src', 'img_title', 'mx4d', 'dolby', 'atmos',
        'mega_only', 'summary', 'score', 'title',
        'booking_rate', 'release_date', 'like'
    ])

    # table.to_csv("megaboxCrawling.csv", encoding="cp949", errors="ignore", index=False)
    table.to_csv("megaboxCrawling.csv", encoding="utf-8-sig", index=False)
    print("megaboxCrawling.csv 파일저장 완료 >>>>>>>>>>>>>>>>>")

    # 4) 드라이버 종료
    wd.quit()


if __name__ == "__main__":
    main()


Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>
총 161개 영화 감지됨
megaboxCrawling.csv 파일저장 완료 >>>>>>>>>>>>>>>>>
